<a href="https://colab.research.google.com/github/AbhijeetKumarThakur2198/Python_Projects/blob/main/Main_Projects/Train_Text_Generation_Model/Train_Text_Generation_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's start training!

In [ ]:

#@title Train Text Generation Model

try:
    import sys # IMPORTANT FOR EXIT IF MODULE NOT FOUND
except Exception as e:
    print(f"Error: {e}")
try:
    import json # IMPORTANT FOR CONFIG FILE
except Exception as e:
    print(f"Error: {e}")
    sys.exit(1)

try:
    import tqdm # TQDM TO SEE PROGRESS BAR (OPTIONAL)
    from tqdm import auto
except ModuleNotFoundError:
    print("tqdm not found; you will not be able to see the training progress bar!")
    pass
except Exception as e:
    print(f"Error: {e}")
    pass

try:
    import torch # NECCESARY FOR TRAINING MODEL
    import torch.nn as nn
    from torch.nn import functional as F
    print("Pytorch found!")
    print("Now we are good to go!")
except ModuleNotFoundError:
    print("Pytorch not found!")
    sys.exit(1)
except Exception as e:
    print(f"Error: {e}")
    sys.exit(1)

#@markdown Fill Hyperparameters
data_path = "/content/input.txt" #@param {type:"string"}

bias = "True" #@param ["True", "False"]

if bias == "True":
    bias = True

elif bias == "False":
    bias = False

num_head = 15 #@param {type:"integer"}
num_layer = 28 #@param {type:"integer"}
batch_size = 12 #@param {type:"integer"}
dimension = 2 #@param {type:"integer"}
split_ratio = 0.8 #@param {type:"number"}
block_size = 512 #@param {type:"integer"}
num_embd = 25 #@param {type:"integer"}
dropout = 0.2 #@param {type:"number"}
learning_rate = 5e-5 #@param {type:"number"}
dataset_type = "Train" #@param ["Train", "Validation"]
weight_init_mean = 0.0 #@param {type:"number"}
validation_interval = 500 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
torch.manual_seed(seed)
standard_deviation = 0.2 #@param {type:"number"}
total_training_steps = 1000 #@param {type:"integer"}
validation_iterations = 200 #@param {type:"integer"}
tensor_dtype = "int_64_signed" #@param ["int_64_signed", "int_8_unsigned"]

if tensor_dtype == "int_64_signed":
    tensor_dtype = torch.int64

elif tensor_dtype == "int_8_unsigned":
    tensor_dtype = torch.uint8

gradient_clip_threshold = 0.2 #@param {type:"number"}
optimizer = "Adamax" #@param ["AdamW", "SGD", "Adagrad", "RMSprop", "Adamax"]

if optimizer == "AdamW":
    optimizer = torch.optim.AdamW

elif optimizer == "SGD":
    optimizer = torch.optim.SGD

elif optimizer == "Adagrad":
    optimizer = torch.optim.Adagrad

elif optimizer == "RMSprop":
    optimizer = torch.optim.RMSprop

elif optimizer == "Adamax":
    optimizer = torch.optim.Adamax

device = 'cuda' #@param ["cuda", "cpu"]

def load_and_process_data(data_path, tensor_dtype=torch.int64, get_split=split_ratio):
    try:
        with open(data_path, 'r', encoding='utf-8') as f:
           data_text = f.read()
    except FileNotFoundError:
       print(f"Error: File not found at {data_path}")
       return None
    except Exception as e:
       print(f"Error: Unable to read file at {data_path} Error: {e}")
       return None

    characters = sorted(list(set(data_text)))
    vocab_size = len(characters)
    string_to_index = {letters: index for index, letters in enumerate(characters)}

    def encode(string):
        return [string_to_index[encoded_characters] for encoded_characters in string]

    encoded_data = torch.tensor(encode(data_text), dtype=tensor_dtype)
    split_index = int(get_split * len(encoded_data))
    train_data = encoded_data[:split_index]
    validation_data = encoded_data[split_index:]
    return train_data, validation_data, vocab_size

def get_batch(dataset_type):
    if dataset_type.lower() == "train":
        data = train_data
    elif dataset_type.lower() == "validation":
        data = validation_data
    else:
        print(f"Unknown dataset_type found: {dataset_type}\nPlease use train or validation only!")

    indices = torch.randint(len(data) - block_size, (batch_size,))
    input_sequence = torch.stack([data[index:index+block_size] for index in indices])
    target_sequence = torch.stack([data[index+1:index+block_size+1] for index in indices])
    input_sequence, target_sequence = input_sequence.to(device), target_sequence.to(device)
    return input_sequence, target_sequence

def make_config_file():
    config = {
    "batch_size": batch_size,
    "block_size": block_size,
    "total_training_steps": total_training_steps,
    "validation_interval": validation_interval,
    "learning_rate": learning_rate,
    "device": device,
    "validation_iterations": validation_iterations,
    "num_embd": num_embd,
    "num_head": num_head,
    "num_layer": num_layer,
    "dropout": dropout,
    "bias": bias,
    "dimension": dimension,
    "standard_deviation": standard_deviation,
    "weight_init_mean": weight_init_mean,
    "gradient_clip_threshold": gradient_clip_threshold,
}
    with open('config.json', 'w') as config_file:
        json.dump(config, config_file, indent=2)

@torch.no_grad()
def estimate_loss():
    losses_dictionary = {}
    model.eval()
    for dataset_type in ['train', 'validation']:
        losses = torch.zeros(validation_iterations)
        for k in range(validation_iterations):
            X, Y = get_batch(dataset_type)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        losses_dictionary[dataset_type] = losses.mean()
    model.train()
    return losses_dictionary

# TRANSFORMER ARCHITECTURE
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(num_embd, head_size, bias=bias)
        self.query = nn.Linear(num_embd, head_size, bias=bias)
        self.value = nn.Linear(num_embd, head_size, bias=bias)
        self.register_buffer('lower_triangular_matrix', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        weights = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        weights = weights.masked_fill(self.lower_triangular_matrix[:T, :T] == 0, float('-inf'))
        weights = F.softmax(weights, dim=dimension)
        weights = self.dropout(weights)
        value = self.value(x)
        attended_values = weights @ value
        return attended_values

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.linear_projection = nn.Linear(head_size * num_heads, num_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        concatenated_output = torch.cat([h(x) for h in self.heads], dimension)
        processed_output = self.dropout(self.linear_projection(concatenated_output))
        return processed_output

class FeedFoward(nn.Module):
    def __init__(self, num_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(num_embd, 4 * num_embd), nn.ReLU(), nn.Linear(4 * num_embd, num_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, num_embd, num_head):
        super().__init__()
        head_size = num_embd // num_head
        self.sa = MultiHeadAttention(num_head, head_size)
        self.ffwd = FeedFoward(num_embd)
        self.ln1 = nn.LayerNorm(num_embd)
        self.ln2 = nn.LayerNorm(num_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TextGenerationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, num_embd)
        self.position_embedding_table = nn.Embedding(block_size, num_embd)
        self.blocks = nn.Sequential(*[Block(num_embd, num_head=num_head) for _ in range(num_layer)])
        self.ln_f = nn.LayerNorm(num_embd)
        self.lm_head = nn.Linear(num_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=weight_init_mean, std=standard_deviation)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=weight_init_mean, std=standard_deviation)

    def forward(self, idx, targets=None):
        idx = idx.int()
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

try:
    train_data, validation_data, vocab_size = load_and_process_data(data_path, tensor_dtype)
    model = TextGenerationModel().to(device)
    make_config_file()
    optimizer = optimizer(model.parameters(), lr=learning_rate)
    print("Your model parameters will be", sum(p.numel() for p in model.parameters()) / 1e6, "Million")
    print("\nThis training process take time it depend what hypermeters you filled. So please wait until it show Training is completed!")
    try:
        bar = auto.tqdm(range(1, total_training_steps + 1), position=0)
    except ImportError:
        print("tqdm not found; progress bar will not be displayed.")
        bar = range(1, total_training_steps + 1)
    for iter in bar:
        if iter % validation_interval == 0 or iter == total_training_steps:

            losses = estimate_loss()
            torch.save(model.state_dict(), f'model{iter}.pth')
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['validation']:.4f}, Checkpoint model saved!")
            xb, yb = get_batch(dataset_type)
            logits, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_threshold)
            loss.backward()
            optimizer.step()
        bar.update(1)
    bar.close()
    print("Training is completed!")

except Exception as e:
    print(f"Error: {e}")